# Land-cover classification with TorchGeo — EuroSAT, pretrained vs from-scratch

The real GPU workflow for this project. It loads **EuroSAT** (Sentinel-2 land-cover patches) through [microsoft/torchgeo](https://github.com/microsoft/torchgeo), fine-tunes a **ResNet18** twice — once from ImageNet-pretrained weights, once from scratch — and evaluates both on a held-out split with **this repo's pure-numpy `lcnet.metrics`**, so the baseline and the deep model are scored by the same code.

**Runtime note.** This needs a GPU. On Colab: *Runtime -> Change runtime type -> GPU*. The code cells are written to be correct and coherent but are not executed in CI (CI only runs the numpy core). EuroSAT downloads automatically on first use (~90 MB).

## 0. Install (Colab)

Uncomment on a fresh Colab GPU runtime.

In [ ]:
# !pip install torch torchvision torchgeo numpy scikit-learn matplotlib
# If running inside the repo, also make `lcnet` importable:
# import sys; sys.path.append('src')

## 1. Load EuroSAT via TorchGeo

EuroSAT ships 27,000 Sentinel-2 patches across 10 land-cover classes. We use the RGB bands so the patches are compatible with the ImageNet-pretrained ResNet stem.

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchgeo.datasets import EuroSAT

SEED = 0
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

dataset = EuroSAT(root="data/eurosat", download=True, bands=EuroSAT.rgb_bands)
num_classes = len(dataset.classes)
print(len(dataset), "patches,", num_classes, "classes")
print(dataset.classes)

n_test = int(0.2 * len(dataset))
n_train = len(dataset) - n_test
gen = torch.Generator().manual_seed(SEED)
train_ds, test_ds = random_split(dataset, [n_train, n_test], generator=gen)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=64)

## 2. Visualise a few patches

A quick look at the imagery and labels before training.

In [ ]:
import matplotlib.pyplot as plt

batch = next(iter(train_dl))
imgs, labels = batch["image"], batch["label"]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, img, lab in zip(axes, imgs[:5], labels[:5]):
    rgb = img.float()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    ax.imshow(rgb.permute(1, 2, 0).numpy())
    ax.set_title(dataset.classes[int(lab)])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. A ResNet18, two ways

Same architecture and training loop; the only difference is whether we start from ImageNet weights. We swap the final layer for the 10 EuroSAT classes.

In [ ]:
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18


def build_model(pretrained: bool):
    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model = resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


def train_model(model, epochs=10, lr=1e-3):
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        running = 0.0
        for b in train_dl:
            x = b["image"].float().to(device)
            y = b["label"].to(device)
            optim.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optim.step()
            running += float(loss)
        print(f"epoch {epoch + 1}/{epochs}  loss {running / len(train_dl):.4f}")
    return model

## 4. Evaluate with this repo's numpy metrics

The whole point of the comparison: collect predictions, then score with the **same** `lcnet.metrics` used for the pure-numpy baseline, so the two are directly comparable.

In [ ]:
import numpy as np

from lcnet.metrics import accuracy, cohen_kappa, confusion_matrix, macro_f1


@torch.no_grad()
def evaluate(model):
    model.eval()
    y_true, y_pred = [], []
    for b in test_dl:
        x = b["image"].float().to(device)
        preds = model(x).argmax(dim=1).cpu().numpy()
        y_pred.extend(int(p) for p in preds)
        y_true.extend(int(t) for t in b["label"].numpy())
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return {
        "accuracy": accuracy(y_true, y_pred),
        "macro_f1": macro_f1(y_true, y_pred, num_classes),
        "cohen_kappa": cohen_kappa(y_true, y_pred, num_classes),
        "confusion": confusion_matrix(y_true, y_pred, num_classes),
        "y_true": y_true,
        "y_pred": y_pred,
    }

## 5. Pretrained vs from-scratch

Train both and compare. On a small dataset and few epochs, ImageNet pretraining usually wins clearly — that gap is the headline result for the model card.

In [ ]:
pre_model = train_model(build_model(pretrained=True), epochs=10)
pre = evaluate(pre_model)
print(
    "pretrained :",
    {k: round(pre[k], 4) for k in ("accuracy", "macro_f1", "cohen_kappa")},
)

scratch_model = train_model(build_model(pretrained=False), epochs=10)
scratch = evaluate(scratch_model)
print(
    "scratch    :",
    {k: round(scratch[k], 4) for k in ("accuracy", "macro_f1", "cohen_kappa")},
)

print("\nFill these into the README model-card table:")
print(f"pretrained  acc={pre['accuracy']:.3f}  macro_f1={pre['macro_f1']:.3f}")
print(f"scratch     acc={scratch['accuracy']:.3f}  macro_f1={scratch['macro_f1']:.3f}")

## 6. Confusion matrix and failure cases

Where does the better model still get confused? The confusion matrix (from `lcnet.metrics`) shows which land-cover classes get swapped; the panel shows individual misclassified patches.

In [ ]:
import matplotlib.pyplot as plt

cm = pre["confusion"]
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(dataset.classes, rotation=90)
ax.set_yticklabels(dataset.classes)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("EuroSAT confusion (pretrained ResNet18)")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Failure cases: test patches the pretrained model got wrong.
wrong = np.where(pre["y_true"] != pre["y_pred"])[0][:5]
test_imgs = [test_ds[i]["image"] for i in wrong]
fig, axes = plt.subplots(1, len(wrong), figsize=(3 * len(wrong), 3))
for ax, idx, img in zip(np.atleast_1d(axes), wrong, test_imgs):
    rgb = img.float()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    ax.imshow(rgb.permute(1, 2, 0).numpy())
    ax.set_title(
        f"t:{dataset.classes[pre['y_true'][idx]]}\np:{dataset.classes[pre['y_pred'][idx]]}",
        fontsize=8,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. The numpy baseline, for context

The same `lcnet` package that scored the deep model also ships a from-scratch, GPU-free softmax baseline. Run it here so both numbers sit side by side.

In [ ]:
from lcnet.demo import run_demo

baseline = run_demo(seed=0, out_dir="outputs")
print("numpy softmax baseline (synthetic EuroSAT-like):")
print(f"  accuracy  {baseline['test_accuracy']:.3f}")
print(f"  macro_f1  {baseline['test_macro_f1']:.3f}")